<a href="https://colab.research.google.com/github/busycaesar/LLM_Eval/blob/Master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets anthropic tqdm pandas

## 0. Configuration

In [ ]:
MODEL = "claude-sonnet-5"
DATASET = "cais/mmlu"
SAMPLE_SIZE = 10

# Parallel requests
MAX_WORKERS = 8

## 1. Load Model

In [ ]:
from google.colab import userdata
from anthropic import Anthropic

client = Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

## 2. Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET, "all", split="test")

if SAMPLE_SIZE:
    dataset = dataset.shuffle(seed=0).select(range(min(SAMPLE_SIZE, len(dataset))))

rows = [dict(r) for r in dataset]
print(f"{len(rows)} items loaded")
print(rows[0])

## 3. Prompt template

In [ ]:
LETTERS = ["A", "B", "C", "D"]

def build_prompt(row):
    choices = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(row["choices"]))
    return (
        "Answer the following multiple choice question.\n\n"
        f"Question: {row['question']}\n\n"
        f"{choices}\n\n"
        "Reply with only the letter of the correct answer."
    )

print(build_prompt(rows[0]))

## 4. Model call with retry and answer extraction function

In [ ]:
import re, time

def call_model(prompt, retries=4):
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=16,
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(b.text for b in resp.content if b.type == "text")
        except Exception as e:
            if attempt == retries - 1:
                return f"ERROR: {e}"
            time.sleep(2 ** attempt)

def extract_letter(text):
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None

## 5. Run the eval

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import pandas

def evaluate_one(row):
    raw_response = call_model(build_prompt(row))
    prediction = extract_letter(raw_response)
    correct_answer = LETTERS[row["answer"]]

    return {
        "subject": row["subject"],
        "question": row["question"],
        "gold": correct_answer,
        "prediction": prediction,
        "raw_response": raw_response,
        "correct": prediction == correct_answer,
    }

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    results = list(tqdm(ex.map(evaluate_one, rows), total=len(rows)))

df = pandas.DataFrame(results)
df.head()

## 6. Score and aggregate

In [ ]:
accuracy = df["correct"].mean()
unparsed = df["prediction"].isna().sum()
errors = df["raw_response"].str.startswith("ERROR:").sum()

by_subject = (
    df.groupby("subject")["correct"]
      .agg(accuracy="mean", n="count")
      .sort_values("accuracy", ascending=False)
)

## 7. Save results

In [ ]:
slug = MODEL.replace("/", "_")
df.to_csv(f"results_{slug}.csv", index=False)
by_subject.to_csv(f"by_subject_{slug}.csv")

print(f"Model:     {MODEL}")
print(f"Dataset:   {DATASET}")
print(f"Items:     {len(df)}")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Unparsed:  {unparsed}")
print(f"API errors: {errors}")